In [1]:
import pandas as pd

base = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"
fs = pd.read_csv(base + "finess_sygen_final.csv", dtype=str)
insee = pd.read_csv(base + "insee_communes_final.csv", dtype=str)

In [2]:
def harmoniser(code):
    if pd.isna(code):
        return code
    if "75101" <= code <= "75120":
        return "75056"   # Paris
    if "69381" <= code <= "69389":
        return "69123"   # Lyon
    if "13201" <= code <= "13216":
        return "13055"   # Marseille
    return code

fs2 = fs.dropna(subset=["CODGEO"]).copy()
fs2["CODGEO_ORIG"] = fs2["CODGEO"]
fs2["CODGEO"] = fs2["CODGEO"].map(harmoniser)
fs3 = fs2[fs2["ETAT"] == "A"].copy()

print(len(fs), len(fs2), len(fs3))

3988 3940 3921


In [3]:
cols_num = ["LIT_MED", "LIT_CHI", "LIT_OBS", "LIT_MCO", "LIT_SMR", "LIT_USLD",
            "NB_SCAN", "NB_IRM", "NB_MAMM"]
for c in cols_num:
    fs3[c] = pd.to_numeric(fs3[c], errors="coerce")

hopitaux = fs3.groupby("CODGEO")[cols_num].sum(min_count=1)
hopitaux["nb_etablissements"] = fs3.groupby("CODGEO")["FI"].count()
hopitaux = hopitaux.reset_index()

print(hopitaux.shape)
print(hopitaux["nb_etablissements"].sum())
print(hopitaux.head())

(1714, 11)
3921
  CODGEO  LIT_MED  LIT_CHI  LIT_OBS  LIT_MCO  LIT_SMR  LIT_USLD  NB_SCAN  \
0  01004     31.0     27.0     15.0     73.0      NaN       NaN      NaN   
1  01033      NaN      NaN      NaN      NaN      NaN       NaN      NaN   
2  01034     56.0     19.0      8.0     83.0     40.0       NaN      1.0   
3  01053     26.0     44.0      NaN     70.0    156.0      11.0      0.0   
4  01082      NaN      NaN      NaN      NaN     70.0       NaN      NaN   

   NB_IRM  NB_MAMM  nb_etablissements  
0     NaN      NaN                  3  
1     NaN      NaN                  1  
2     0.0      1.0                  2  
3     0.0      0.0                  5  
4     NaN      NaN                  1  


## Partie 2 : fusion avec les données INSEE

In [4]:
# communes avec établissement absentes de l'INSEE 
print(sorted(set(hopitaux["CODGEO"]) - set(insee["CODGEO"])))

fusion = insee.merge(hopitaux, on="CODGEO", how="left", validate="one_to_one")
fusion["nb_etablissements"] = fusion["nb_etablissements"].fillna(0)

print(fusion.shape)
print("communes avec au moins 1 établissement :", (fusion["nb_etablissements"] > 0).sum())
print(fusion[fusion["CODGEO"] == "01004"].T)

['97123', '97127', '97502', '97606', '97611', '97615']
(34858, 45)
communes avec au moins 1 établissement : 1708
                                                2
CODGEO                                      01004
LIBGEO                          Ambérieu-en-Bugey
SEXE1_AGEPYR1000                 291.774098436508
SEXE1_AGEPYR1003                 327.193219345162
SEXE1_AGEPYR1006                 525.370991705619
SEXE1_AGEPYR1011                 719.637561687717
SEXE1_AGEPYR1018                 733.735171978173
SEXE1_AGEPYR1025                 1581.24638491537
SEXE1_AGEPYR1040                 1344.79206887329
SEXE1_AGEPYR1055                 833.964092706198
SEXE1_AGEPYR1065                 888.674859386373
SEXE1_AGEPYR1080                 244.547759254982
SEXE2_AGEPYR1000                 323.969423155764
SEXE2_AGEPYR1003                 283.369968380778
SEXE2_AGEPYR1006                  467.59143190407
SEXE2_AGEPYR1011                 664.536778087955
SEXE2_AGEPYR1018                  736

In [5]:
print("établissements dans la fusion :", fusion["nb_etablissements"].sum(), "sur", len(fs3))
print("codes 97 dans l'INSEE :", insee["CODGEO"].str.startswith("97").sum())
print(fs3[fs3["CODGEO"].isin(['97123', '97127', '97502', '97606', '97611', '97615'])][["CODGEO", "NOM", "CATEGORIE"]])


établissements dans la fusion : 3906.0 sur 3921
codes 97 dans l'INSEE : 112
     CODGEO                                                NOM CATEGORIE
3853  97123  CENTRE HOSPITALIER IRENEE DE BRUYN EX HÔPITAL ...       106
3855  97127          CENTRE HOSPITALIER LOUIS CONSTANT FLEMING       355
3868  97127                      ANTENNE AUDRA DE SAINT MARTIN       146
3875  97127                                  HAD  ILES DU NORD       127
3879  97123  CH LOUIS CONSTANT FLEMING - ANTENNE SMUR DE SA...       355
3880  97123            CLINIQUE DE CHOISY HAD SAINT BARTHELEMY       127
3881  97127                                    CLINIQUE WATAKI       109
3980  97502                 CENTRE HOSPITALIER  FRANCOIS DUNAN       355
3981  97611                      CENTRE HOSPITALIER DE MAYOTTE       355
3982  97611               UNITE AUTODIALYSE ASSISTEE MAMOUDZOU       141
3983  97611                                         UDM KAWENI       141
3984  97606                   UNITE D'AUTODIALYS

In [6]:
outre_mer = insee[insee["CODGEO"].str.startswith("97")]
print(len(outre_mer))
print(outre_mer["CODGEO"].str[:3].value_counts().sort_index())

112
CODGEO
971    32
972    34
973    22
974    24
Name: count, dtype: int64


**Périmètre.** 15 établissements actifs (0,4 % du total), situés à Saint-Martin, Saint-Barthélemy, Saint-Pierre-et-Miquelon et Mayotte, sont exclus : ces territoires sont absents des tables INSEE de population, de revenus et de densité, ce qui empêche de calculer un indicateur pour eux. L'analyse couvre la France métropolitaine et les quatre départements d'outre-mer historiques (Guadeloupe, Martinique, Guyane, La Réunion)

In [7]:
hors_insee = hopitaux[~hopitaux["CODGEO"].isin(insee["CODGEO"])]
print(len(hors_insee), "communes hors table INSEE,",
      int(hors_insee["nb_etablissements"].sum()), "établissements exclus")

6 communes hors table INSEE, 15 établissements exclus


In [8]:
fusion["nb_etablissements"] = fusion["nb_etablissements"].fillna(0).astype(int)

In [9]:
print(fusion["nb_etablissements"].dtype)
print(fusion["nb_etablissements"].isna().sum())
print((fusion["nb_etablissements"] == 0).sum())

int64
0
33150


In [10]:
fusion.head()

,CODGEO,LIBGEO,SEXE1_AGEPYR1000,SEXE1_AGEPYR1003,SEXE1_AGEPYR1006,SEXE1_AGEPYR1011,SEXE1_AGEPYR1018,SEXE1_AGEPYR1025,SEXE1_AGEPYR1040,SEXE1_AGEPYR1055,...,LIT_MED,LIT_CHI,LIT_OBS,LIT_MCO,LIT_SMR,LIT_USLD,NB_SCAN,NB_IRM,NB_MAMM,nb_etablissements
0,01001,L'Abergement-Clémenciat,19.0,15.0,36.0,42.0,15.0,77.0,87.0,66.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,01002,L'Abergement-de-Varey,3.0,6.0,7.0,22.0,6.0,20.0,39.0,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,01004,Ambérieu-en-Bugey,291.774098436508,327.193219345162,525.370991705619,719.637561687717,733.735171978173,1581.24638491537,1344.79206887329,833.964092706198,...,31.0,27.0,15.0,73.0,NaN,NaN,NaN,NaN,NaN,3
3,01005,Ambérieux-en-Dombes,36.2624041600494,37.108922388434,65.9432761131205,80.0820466018419,54.4457784678517,193.448597496766,196.440979404838,137.738813251787,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,01006,Ambléon,2.0,1.0,2.0,1.0,4.0,12.0,11.0,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [11]:

import os

chemin = "s3://emma457/Projet_python/fusion_insee_hopitaux.csv"
opts = {"client_kwargs": {"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]}}

fusion.to_csv(
    chemin,
    index=False,
    storage_options=opts
)

print("Fichier sauvegardé :", chemin)


PermissionError: Access Denied.

Etudes par densité des communautés

In [12]:
dens = pd.read_csv(base + "densite_communes_2025.csv", dtype=str)
cols_dens = ["DENS", "LIBDENS", "DENS7", "LIBDENS7"]

fusion = fusion.drop(columns=cols_dens).merge(
    dens[["CODGEO"] + cols_dens], on="CODGEO", how="left", validate="one_to_one"
)
print(fusion["LIBDENS"].isna().sum(), fusion.shape)   # attendu : 0 (34858, 23)

0 (34858, 45)


Regardons par population

In [ ]:
#fusion["a_etablissement"] = fusion["nb_etablissements"] > 0
#fusion["population"] = pd.to_numeric(fusion["population"], errors="coerce")

#tableau = fusion.groupby("LIBDENS", dropna=False)["a_etablissement"].agg(
    #nb_communes="count", part_communes_equipees="mean")
#tableau["part_communes_equipees"] = (tableau["part_communes_equipees"] * 100).round(1)

#pop_tot = fusion.groupby("LIBDENS", dropna=False)["population"].sum()
#pop_avec = fusion[fusion["a_etablissement"]].groupby("LIBDENS", dropna=False)["population"].sum()
#tableau["part_population_desservie"] = (pop_avec / pop_tot * 100).round(1)

#print(tableau)

SyntaxError: unmatched ')' (1315846186.py, line 6)

## Partie 3 : fusion avec l'offre de soins de ville (RPPS)

In [13]:

# Chargement de l'offre de soins de ville déjà agrégée par commune
rpps = pd.read_csv(
    base + "offre_soins_commune.csv",
    dtype={"Code commune (coord. structure)": str}
)

print(rpps.shape)
print(rpps.head())


(17213, 3)
  Code commune (coord. structure) Libellé commune (coord. structure)  \
0                           00000                            Inconnu   
1                           01001            L'Abergement-Clémenciat   
2                           01002              L'Abergement-de-Varey   
3                           01004                  Ambérieu-en-Bugey   
4                           01005                Ambérieux-en-Dombes   

   nb_professionnels  
0                  9  
1                  1  
2                  1  
3                521  
4                 29  



La table RPPS est déjà agrégée au niveau communal.

On harmonise son code commune avec le même traitement que celui utilisé plus haut pour FINESS-SYGEN, notamment pour les arrondissements de Paris, Lyon et Marseille.


In [14]:

rpps = rpps.rename(
    columns={
        "Code commune (coord. structure)": "CODGEO",
        "Libellé commune (coord. structure)": "LIBGEO_RPPS"
    }
)

rpps = rpps.dropna(subset=["CODGEO"]).copy()

# Même harmonisation géographique que pour FINESS-SYGEN
rpps["CODGEO_ORIG"] = rpps["CODGEO"]
rpps["CODGEO"] = rpps["CODGEO"].map(harmoniser)

rpps["nb_professionnels"] = pd.to_numeric(
    rpps["nb_professionnels"],
    errors="coerce"
)

print("Communes distinctes avant harmonisation :", rpps["CODGEO_ORIG"].nunique())
print("Communes distinctes après harmonisation :", rpps["CODGEO"].nunique())


Communes distinctes avant harmonisation : 17213
Communes distinctes après harmonisation : 17168



Après harmonisation, plusieurs lignes peuvent désormais correspondre à une même commune, notamment pour Paris, Lyon et Marseille.

On réagrège donc l'offre au niveau du `CODGEO` harmonisé avant la fusion finale.


In [15]:

rpps_commune = (
    rpps
    .groupby("CODGEO", as_index=False)
    .agg(
        nb_professionnels=("nb_professionnels", "sum")
    )
)

print(rpps_commune.shape)
print("Doublons CODGEO :", rpps_commune["CODGEO"].duplicated().sum())


(17168, 2)
Doublons CODGEO : 0


## Contrôle de la couverture géographique du RPPS

In [16]:

codes_insee = set(fusion["CODGEO"].dropna())
codes_rpps = set(rpps_commune["CODGEO"].dropna())

print("Communes dans la base INSEE + hôpitaux :", len(codes_insee))
print("Communes dans le RPPS :", len(codes_rpps))
print("Communes INSEE sans ligne RPPS :", len(codes_insee - codes_rpps))
print("Codes RPPS absents de l'INSEE :", len(codes_rpps - codes_insee))

print("\nExemples de codes RPPS hors INSEE :")
print(sorted(codes_rpps - codes_insee)[:20])


Communes dans la base INSEE + hôpitaux : 34858
Communes dans le RPPS : 17168
Communes INSEE sans ligne RPPS : 18143
Codes RPPS absents de l'INSEE : 453

Exemples de codes RPPS hors INSEE :
['00000', '01055', '01091', '01154', '01330', '02077', '02300', '02630', '02700', '03153', '04056', '04196', '04198', '05002', '05069', '05143', '05175', '08443', '12076', '12114']


## Fusion finale avec l'offre de soins de ville

In [17]:

base_finale = fusion.merge(
    rpps_commune,
    on="CODGEO",
    how="left",
    validate="one_to_one"
)

# Une commune sans ligne dans l'agrégat RPPS reçoit 0 professionnel compté.
base_finale["nb_professionnels"] = (
    base_finale["nb_professionnels"]
    .fillna(0)
    .astype(int)
)

print("Dimensions finales :", base_finale.shape)
print("Communes distinctes :", base_finale["CODGEO"].nunique())
print("Doublons CODGEO :", base_finale["CODGEO"].duplicated().sum())

assert len(base_finale) == len(fusion)
assert not base_finale["CODGEO"].duplicated().any()


Dimensions finales : (34858, 46)
Communes distinctes : 34858
Doublons CODGEO : 0


## Vérifications finales

In [18]:

colonnes_controle = [
    "CODGEO",
    "LIBGEO",
    "nb_professionnels",
    "nb_etablissements",
    "LIT_MCO",
    "LIT_SMR",
    "NB_SCAN",
    "NB_IRM"
]

colonnes_controle = [
    col for col in colonnes_controle
    if col in base_finale.columns
]

print("Dimensions :", base_finale.shape)
print("\nValeurs manquantes :")
print(base_finale[colonnes_controle].isna().sum())

base_finale.head()


Dimensions : (34858, 46)

Valeurs manquantes :
CODGEO                   0
LIBGEO                   0
nb_professionnels        0
nb_etablissements        0
LIT_MCO              33947
LIT_SMR              33587
NB_SCAN              34111
NB_IRM               34111
dtype: int64


,CODGEO,LIBGEO,SEXE1_AGEPYR1000,SEXE1_AGEPYR1003,SEXE1_AGEPYR1006,SEXE1_AGEPYR1011,SEXE1_AGEPYR1018,SEXE1_AGEPYR1025,SEXE1_AGEPYR1040,SEXE1_AGEPYR1055,...,LIT_USLD,NB_SCAN,NB_IRM,NB_MAMM,nb_etablissements,DENS,LIBDENS,DENS7,LIBDENS7,nb_professionnels
0,01001,L'Abergement-Clémenciat,19.0,15.0,36.0,42.0,15.0,77.0,87.0,66.0,...,NaN,NaN,NaN,NaN,0,3,Rural,6,Rural à habitat dispersé,1
1,01002,L'Abergement-de-Varey,3.0,6.0,7.0,22.0,6.0,20.0,39.0,12.0,...,NaN,NaN,NaN,NaN,0,3,Rural,6,Rural à habitat dispersé,1
2,01004,Ambérieu-en-Bugey,291.774098436508,327.193219345162,525.370991705619,719.637561687717,733.735171978173,1581.24638491537,1344.79206887329,833.964092706198,...,NaN,NaN,NaN,NaN,3,2,Urbain intermédiaire,2,Centres urbains intermédiaires,521
3,01005,Ambérieux-en-Dombes,36.2624041600494,37.108922388434,65.9432761131205,80.0820466018419,54.4457784678517,193.448597496766,196.440979404838,137.738813251787,...,NaN,NaN,NaN,NaN,0,3,Rural,5,Bourgs ruraux,29
4,01006,Ambléon,2.0,1.0,2.0,1.0,4.0,12.0,11.0,14.0,...,NaN,NaN,NaN,NaN,0,3,Rural,6,Rural à habitat dispersé,0


## Sauvegarde de la base communale finale

In [19]:

output_path = "s3://emma457/Projet_python/base_communes_finale.csv"

base_finale.to_csv(
    output_path,
    index=False,
    storage_options=opts
)

print("Table finale enregistrée :", output_path)


KeyboardInterrupt: 


## Résultat

La base finale part désormais directement de la fusion construite précédemment :

```text
INSEE + FINESS-SYGEN
```

puis ajoute :

```text
+ RPPS
```

On évite ainsi de refaire l'agrégation FINESS-SYGEN déjà réalisée dans ce notebook.
